In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 8.1 The Many-Electron Problem

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VIII — Electronic Structure and Many-Body Matter",
    number="8.1",
    title="The Many-Electron Problem",
    blurb="The problem that consumes more computer cycles than any other question "
    "science asks, stated honestly before anyone tries to solve it. We derive the "
    "atomic units that strip the clutter from every equation to come, certify the "
    "radial machinery of the volume on hydrogen, measure the exponential wall that "
    "makes the exact problem unsolvable for even five electrons, carry out the "
    "Born–Oppenheimer separation on a molecule we can actually diagonalize, "
    "and watch a two-parameter variational guess for helium miss the exact energy "
    "by a gap with a name: correlation.",
    difficulty="advanced",
    estimate="110–140 min",
)

## Notebook overview

Volume VIII opens with a confession the previous volumes never had to make. Volume VI
solved one particle at a time; Volume VII took many particles and let them ignore each
other, hiding every interaction inside a chemical potential. But the silicon running
this page, the metal in the machine's frame, the molecules in the reader: all of it is
electrons interacting through the Coulomb force, and the honest governing equation is a
Schrödinger equation in $3N$ coordinates that no computer will ever store, let alone
solve, for $N$ beyond a handful. This notebook states that problem precisely and
measures why it is hard, because every method in the sixteen notebooks that follow is a
strategy for evading a wall we will first run into deliberately.

The plan is concrete. We derive Hartree atomic units and let `scipy.constants` confirm
that one Hartree is $27.211$ eV, so every later equation sheds its constants. We
certify the volume's radial workhorse (the tridiagonal eigensolver of
[§6.10](../06-quantum-mechanics/schrodinger-on-a-computer.ipynb), reduced to the
half-line as in [§6.16](../06-quantum-mechanics/central-potentials-3d.ipynb)) on the
one atom whose answers are exact. We tabulate the storage cost of the exact
wavefunction and watch it pass the total RAM on Earth at four electrons. We then carry
out the Born–Oppenheimer separation, not as an assertion but as a computation, on a
one-dimensional model molecule whose electronic curve we diagonalize at every clamped
nuclear distance, and we measure the two small numbers (a vibrational-to-electronic
frequency ratio, a non-adiabatic coupling) that make the separation quantitatively
excellent. The notebook closes on helium: a screened-charge variational guess whose
optimal $Z_{\mathrm{eff}} = 27/16$ arrives in closed form, and whose remaining error
against the exact energy *defines* the quantity, correlation energy, that Volumes
VIII's methods will spend the next sixteen notebooks chasing.

> **Conventions (this notebook and the volume).** Hartree atomic units throughout:
> $\hbar = m_e = e = 4\pi\varepsilon_0 = 1$, so lengths are in Bohr radii
> ($a_0 = 0.529$ Å), energies in Hartree ($E_h = 27.211$ eV), and the hydrogen ground
> state sits at exactly $-1/2$. Grids state their extent and spacing; every
> eigensolver names its routine. The exercises use
> `scipy.linalg.eigh_tridiagonal` for one-dimensional and radial problems, exactly the
> machinery certified in [§6.10](../06-quantum-mechanics/schrodinger-on-a-computer.ipynb).
>
> **How to read the checks.** Each exercise ends with a `validate` call against an
> independent fact: a CODATA constant, an exact eigenvalue, a closed-form optimum, a
> dissociation limit. A ✓ is strong evidence the computation is right; a ✗ is a prompt
> to locate the discrepancy (a genuine error, a convention mismatch, or a tolerance),
> not an automatic verdict.
>
> **Scope.** This notebook states the many-electron problem and its cost; it solves
> nothing larger than two electrons. The systematic attacks begin in
> [§8.2](exact-laboratory.ipynb) (an exactly solvable two-electron laboratory) and
> [§8.3](hartree-fock-atoms.ipynb) (Hartree–Fock). Derivations we sketch are carried
> out in full in Martin, *Electronic Structure* {cite}`martin2004`, Ch. 2–3, and
> Szabo & Ostlund {cite}`szabo1996`, Ch. 1–2.

## Theory in brief

### The Hamiltonian, and the units that clean it

For $N$ electrons at positions $\mathbf r_i$ and $M$ nuclei of charge $Z_k$ at
$\mathbf R_k$, non-relativistic quantum mechanics poses one operator. In SI units it
drips with constants; the cure, standard since the earliest atomic calculations
(Martin {cite}`martin2004`, App. A), is to measure mass in electron masses, charge in
elementary charges, action in $\hbar$, and permittivity in $4\pi\varepsilon_0$. Every
combination then collapses to one, and the natural length and energy scales

```{math}
:label: eq-me-units
a_0 = \frac{4\pi\varepsilon_0\hbar^2}{m_e e^2} = 0.529\,\text{Å},
\qquad
E_h = \frac{m_e e^4}{(4\pi\varepsilon_0)^2\hbar^2} = 27.211\,\text{eV}
```

emerge as the Bohr radius and the Hartree. In these units the full molecular
Hamiltonian reads

```{math}
:label: eq-me-hamiltonian
\hat H \;=\;
-\sum_{i=1}^{N} \frac{\nabla_i^2}{2}
\;-\; \sum_{i,k} \frac{Z_k}{|\mathbf r_i - \mathbf R_k|}
\;+\; \sum_{i<j} \frac{1}{|\mathbf r_i - \mathbf r_j|}
\;-\; \sum_{k} \frac{\nabla_k^2}{2 M_k}
\;+\; \sum_{k<l} \frac{Z_k Z_l}{|\mathbf R_k - \mathbf R_l|},
```

with $M_k$ the nuclear masses in electron masses (a proton is $1836$). Five terms:
electronic kinetic energy, electron–nucleus attraction, electron–electron repulsion,
nuclear kinetic energy, nucleus–nucleus repulsion. The third term is the villain of
this volume. Without it, Eq. {eq}`eq-me-hamiltonian` separates into one-electron
problems and Volume VI has already solved them; with it, the eigenfunction
$\Psi(\mathbf r_1, \dots, \mathbf r_N)$ couples every coordinate to every other and
lives in $3N$ dimensions.

### The wall

How expensive is it to just *store* $\Psi$? A modest grid of $200$ points per
coordinate needs $200^{3N}$ complex numbers. The count is worth tabulating rather than
waving at, because its slope is the whole story: each additional electron multiplies
the cost by $200^3 = 8\times 10^6$. This exponential growth of Hilbert space, the
curse of dimensionality, is why electronic-structure theory exists as a field. Every
method from Hartree–Fock ([§8.3](hartree-fock-atoms.ipynb)) to density-functional
theory ([§8.7](kohn-sham.ipynb)) is a scheme for never touching the full $\Psi$.

### Born–Oppenheimer: freeze the nuclei, then let them move

The first simplification is not about electrons at all. Nuclei are thousands of times
heavier than electrons, so their motion is slow and the electrons follow it
adiabatically. Formally (Born and Oppenheimer 1927; the modern form is in Martin
{cite}`martin2004`, Ch. 3) one solves the *clamped-nuclei* electronic problem at every
fixed nuclear geometry $\mathbf R$,

```{math}
:label: eq-me-bo
\hat H_{\mathrm{el}}(\mathbf R)\,\phi_n(\mathbf r; \mathbf R)
= \varepsilon_n(\mathbf R)\,\phi_n(\mathbf r; \mathbf R),
```

and the electronic eigenvalue $\varepsilon_0(\mathbf R)$, plus the nuclear repulsion,
becomes the potential-energy surface on which the nuclei subsequently move. The
neglected terms couple different electronic surfaces $\phi_n$ through nuclear
derivatives $\partial\phi_n/\partial\mathbf R$, and their size is governed by the mass
ratio: the vibrational quantum $\omega \sim \sqrt{k/\mu}$ is smaller than electronic
gaps by roughly $(m_e/\mu)^{1/2} \approx 1/30$ for hydrogen (with $\mu$ the reduced
nuclear mass, $M_p/2$ for a proton pair), and the direct
non-adiabatic matrix elements are smaller still. Exercises 4 and 5 compute the surface
and then *measure* both small numbers on a model where nothing needs to be taken on
faith.

### The variational principle, and what it cannot reach

The variational theorem of [§6.22](../06-quantum-mechanics/variational-method.ipynb)
survives unchanged in many-electron space: for any normalized trial state $\Phi$,
$\langle\Phi|\hat H|\Phi\rangle \ge E_0$. It is the engine of nearly every method in
this volume. Its first many-electron outing, helium with a screened hydrogenic product
(a calculation older than the Schrödinger equation's second birthday; Szabo & Ostlund
{cite}`szabo1996`, §1.3, work the integrals), gives the closed-form energy curve

```{math}
:label: eq-me-hylleraas
E(Z) \;=\; Z^2 - \tfrac{27}{8}\,Z
\qquad\text{(Hartree)},
```

whose minimum $Z_{\mathrm{eff}} = 27/16 \approx 1.69$ says each electron sees the
nucleus screened by about a third of the other electron's charge. The minimized
energy, $-2.8477$, misses the exact $-2.9037$ {cite}`parryang1989`. That gap is not a
numerical artifact to be converged away: no product of one-electron orbitals, however
optimized, can describe two electrons *avoiding each other*. The missing physics is
electron correlation, and its precise definition (against the Hartree–Fock limit,
$-2.8617$ for helium) arrives in [§8.3](hartree-fock-atoms.ipynb). Volume VIII is,
from here on, the story of that gap.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh_tridiagonal
from scipy.optimize import minimize_scalar
from scipy.constants import (
    physical_constants,
    hbar as HBAR_SI,  # reduced Planck constant, J s
    m_e as ME_SI,  # electron mass, kg
    e as QE_SI,  # elementary charge, C
    epsilon_0 as EPS0_SI,  # vacuum permittivity, F/m
    electron_volt as EV_SI,  # 1 eV in J
)

from ecp import validate
from ecp import draw

INK, AMBER, SOFT = "#16213e", "#c0851a", "#46506b"

# CODATA reference values for the derived unit checks (J and m)
HARTREE_SI = physical_constants["Hartree energy"][0]
BOHR_SI = physical_constants["Bohr radius"][0]

# proton mass in electron masses, for the Born-Oppenheimer mass ratio
MP_OVER_ME = physical_constants["proton-electron mass ratio"][0]


def soft_coulomb(x, center, soft=1.0):
    """Softened Coulomb attraction -1/sqrt((x - center)^2 + soft^2).

    The standard one-dimensional stand-in for the 3-D Coulomb potential: it
    keeps the long-range -1/|x| tail that makes atoms atoms, but caps the
    singularity so a uniform grid can represent the ground state. Used
    throughout Volume VIII's one-dimensional laboratories.

    Parameters
    ----------
    x : numpy.ndarray
        Grid coordinates in Bohr.
    center : float
        Position of the attracting center in Bohr.
    soft : float, optional
        Softening length in Bohr (default 1.0, the conventional choice).

    Returns
    -------
    numpy.ndarray
        The potential on the grid, in Hartree.
    """
    return -1.0 / np.sqrt((x - center) ** 2 + soft**2)


def grid_levels(x, v, k):
    """Lowest k eigenpairs of -(1/2) d^2/dx^2 + v(x) on a uniform grid.

    Discretizes the kinetic term with the three-point Laplacian and solves the
    resulting real symmetric tridiagonal problem with
    scipy.linalg.eigh_tridiagonal, the volume's certified one-dimensional
    workhorse (the machinery of section 6.10). Hard-wall boundaries at the grid
    ends are implicit in the truncation.

    Parameters
    ----------
    x : numpy.ndarray
        Uniform grid (Bohr); the spacing sets the kinetic scale.
    v : numpy.ndarray
        Potential sampled on the grid (Hartree).
    k : int
        Number of lowest eigenpairs to return.

    Returns
    -------
    tuple
        ``(energies, states)``: shape (k,) energies in Hartree and an
        (npts, k) array of eigenvectors normalized so that
        sum(|psi|^2) * dx = 1.
    """
    h = x[1] - x[0]
    energies, states = eigh_tridiagonal(
        1.0 / h**2 + v,
        -0.5 / h**2 * np.ones(len(x) - 1),
        select="i",
        select_range=(0, k - 1),
    )
    return energies, states / np.sqrt(h)

## Exercise 1 — Atomic units from the constants themselves

Equation {eq}`eq-me-units` asserts that the four defining constants of the electron's
world combine into one length and one energy. The assertion deserves a check with real
numbers, and running it once, here, is what licenses every later notebook to write
equations with no constants at all. It also fixes the conversion factors the volume
will use whenever a result must face an experiment quoted in eV or Å.

**Part a)** Build the Bohr radius $a_0 = 4\pi\varepsilon_0\hbar^2/(m_e e^2)$ and the
Hartree $E_h = m_e e^4 / \left[(4\pi\varepsilon_0)^2\hbar^2\right]$ from the
`scipy.constants` values `hbar`, `m_e`, `e`, and `epsilon_0` by direct arithmetic
(products and quotients of the imported floats), and express $E_h$ in eV by dividing
by `scipy.constants.electron_volt`. Print both alongside the CODATA reference entries
`physical_constants["Bohr radius"]` and `physical_constants["Hartree energy"]`.

**Part b)** Two sanity conversions the volume will lean on: the hydrogen binding
energy $\tfrac12 E_h$ in eV (the Rydberg, $13.606$ eV), and room temperature
$k_BT$ at $300$ K expressed in Hartree (use `scipy.constants.k`), the number that
will later say band gaps are enormous compared to thermal energies. Compute both by
the same direct arithmetic.

In [ ]:
# (solution hidden on the public site)


### Validation 1 — the derived units meet CODATA

The built $a_0$ and $E_h$ must match the CODATA entries essentially to machine
precision (they are defined by the same arithmetic), the Hartree must be $27.211$ eV,
and the Rydberg $13.606$ eV.

In [ ]:
validate.close(a0_built, BOHR_SI, "Bohr radius from the defining constants", rtol=1e-9)
validate.close(
    hartree_built, HARTREE_SI, "Hartree energy from the defining constants", rtol=1e-9
)
validate.close(hartree_eV, 27.211386, "one Hartree in eV", rtol=1e-6)
validate.close(rydberg_eV, 13.605693, "the Rydberg in eV", rtol=1e-6)

## Exercise 2 — Certifying the radial machinery on hydrogen

The volume's atoms are spherical, and every radial problem in it reduces (the
$u = rR$ substitution of [§6.16](../06-quantum-mechanics/central-potentials-3d.ipynb))
to a one-dimensional Schrödinger equation on the half-line,

```{math}
:label: eq-me-radial
-\tfrac12\, u''(r) + \left[\, v(r) + \frac{l(l+1)}{2r^2} \right] u(r)
= \varepsilon\, u(r),
\qquad u(0) = 0 ,
```

which a uniform grid $r_j = j\,h$ turns into a symmetric tridiagonal eigenproblem
(the boundary $u(0)=0$ is built in by starting the grid at $r_1 = h$). Before this
machinery computes anything unknown, it must reproduce the one atom whose spectrum is
exact: hydrogen, $v = -1/r$, with $\varepsilon_n = -1/(2n^2)$ and
$\langle r\rangle_{1s} = 3/2$ in atomic units
([§6.17](../06-quantum-mechanics/hydrogen-atom.ipynb) derived all of this; here it is
the *certificate* for the tool).

**Part a)** On the radial grid $r_j = j\,h$ with $h = 0.005$ and $r_{\max} = 60$
(12 000 points), assemble the $l = 0$ tridiagonal matrix for $v(r) = -1/r$ (diagonal
$1/h^2 + v(r_j)$, off-diagonal $-1/(2h^2)$) and obtain the two lowest eigenpairs with
`scipy.linalg.eigh_tridiagonal` (`select="i"`). Report $\varepsilon_{1s}$ and
$\varepsilon_{2s}$ against $-1/2$ and $-1/8$.

**Part b)** Normalize $u_{1s}$ by `numpy.trapezoid` and compute
$\langle r\rangle = \int r\,u_{1s}^2\,dr$ (again `numpy.trapezoid`), which must come
out $3/2$. Plot $u_{1s}$ and $u_{2s}$.

In [ ]:
# (solution hidden on the public site)


### Validation 2 — hydrogen is exact

The grid eigenvalues must match $-1/2$ and $-1/8$ Hartree, and the first moment must
be $3/2$ Bohr, all within the $O(h^2)$ discretization error of the stated grid.

In [ ]:
validate.close(eps_h[0], -0.5, "hydrogen 1s energy on the radial grid", rtol=1e-4)
validate.close(eps_h[1], -0.125, "hydrogen 2s energy on the radial grid", rtol=1e-4)
validate.close(r_mean, 1.5, "hydrogen <r> in the 1s state", rtol=1e-4)

## Exercise 3 — The wall, measured

Storing the exact $\Psi(\mathbf r_1,\dots,\mathbf r_N)$ on a grid of $200$ points per
coordinate costs $200^{3N}$ complex numbers at 16 bytes each. The claim in the theory
section was that this passes planetary resources almost immediately; a claim about
numbers should be a table. For scale, the total installed RAM on Earth is of order
$10^{23}$ bytes (a generous estimate: ten billion devices at ten terabytes each).

**Part a)** Tabulate the storage in bytes for $N = 1,\dots,6$ electrons (compute
$16 \times 200^{3N}$ in floating point via `numpy` powers; at these sizes integers
would overflow C longs, which is itself part of the lesson). Print the table with the
$N$ at which the cost first exceeds $10^{23}$ bytes flagged.

**Part b)** Plot $\log_{10}(\text{bytes})$ against $N$ (a `matplotlib` line with
markers) with a horizontal line at the RAM-on-Earth estimate, and verify by
`numpy.polyfit` (degree 1) that the slope is $3\log_{10}200 \approx 6.9$ decades per
electron: the exponential wall as a measured straight line.

In [ ]:
# (solution hidden on the public site)


### Validation 3 — the wall has the predicted slope

The fitted slope must equal $3\log_{10}200$ (it is exact arithmetic, so the tolerance
is tight), and the crossing of the RAM-on-Earth line must fall between $N = 3$ and
$N = 4$.

In [ ]:
validate.close(
    slope, 3 * np.log10(200.0), "storage slope in decades per electron", rtol=1e-10
)
validate.check(
    bytes_needed[2] < RAM_ON_EARTH < bytes_needed[3],
    "the exact wavefunction passes all RAM on Earth between N = 3 and N = 4",
    f"N=3: {bytes_needed[2]:.1e} B, N=4: {bytes_needed[3]:.1e} B",
)

## Exercise 4 — Born–Oppenheimer, carried out: the electronic curve

The Born–Oppenheimer recipe of Eq. {eq}`eq-me-bo` is usually *described*; on a
one-dimensional model it can simply be *done*. The model molecule is the
soft-Coulomb H$_2^+$: one electron on the line, two protons clamped at
$\pm R/2$, with the electron–nucleus attraction softened as in the Setup helper,

```{math}
:label: eq-me-h2p
\hat H_{\mathrm{el}}(R) = -\tfrac12 \frac{d^2}{dx^2}
- \frac{1}{\sqrt{(x - R/2)^2 + 1}}
- \frac{1}{\sqrt{(x + R/2)^2 + 1}},
\qquad
E(R) = \varepsilon_0(R) + \frac{1}{R} .
```

Diagonalizing $\hat H_{\mathrm{el}}$ at each clamped $R$ and adding the bare nuclear
repulsion $1/R$ gives the molecule's potential-energy curve. Two physical checks are
built into its shape. At large $R$ the electron settles into one well
($\varepsilon_{\mathrm{1well}} = -0.6698$ for this potential) but still feels the
*other* proton's tail $-1/\sqrt{R^2+1} \approx -1/R$, which cancels the nuclear
repulsion: $E(R) \to \varepsilon_{\mathrm{1well}}$, the energy of a neutral atom plus
a distant bare proton. And at intermediate $R$ the shared electron binds the pair
into a molecule: a genuine minimum below the dissociation limit.

In [ ]:
# (solution hidden on the public site)


**Part a)** On the grid $x \in [-20, 20]$ with 2001 points, compute the two lowest
electronic eigenvalues $\varepsilon_0(R), \varepsilon_1(R)$ of
Eq. {eq}`eq-me-h2p` with the Setup helper `grid_levels` (which wraps
`scipy.linalg.eigh_tridiagonal`) at each of 120 clamped separations
$R \in [0.2, 12]$, and form $E(R) = \varepsilon_0(R) + 1/R$.

**Part b)** Locate the equilibrium bond length: the `numpy.argmin` of $E(R)$, refined
by a degree-2 `numpy.polyfit` through the seven surrounding points (vertex of the
parabola). Report $R_{\mathrm{eq}}$ and the binding energy
$E(R_{\mathrm{eq}}) - \varepsilon_{\mathrm{1well}}$, with
$\varepsilon_{\mathrm{1well}}$ computed on the same grid from the single-well
potential.

**Part c)** Plot $E(R)$ with the dissociation asymptote and both electronic surfaces
$\varepsilon_{0,1}(R) + 1/R$: the figure every quantum-chemistry paper draws, here
computed from scratch.

In [ ]:
# (solution hidden on the public site)


### Validation 4 — a bound molecule that dissociates correctly

Three facts of the computed surface: the parabola-refined equilibrium bond sits at
$R_{\mathrm{eq}} = 2.62$ Bohr with binding $-0.111$ Ha (the values measured on this
exact grid), and the curve at $R = 12$ has already returned to the dissociation limit
to better than a percent of the well depth: the large-$R$ cancellation between
nuclear repulsion and the second proton's attraction, working.

In [ ]:
validate.close(R_eq, 2.62, "equilibrium bond length of the model molecule", rtol=1e-2)
validate.close(binding, -0.1106, "binding energy below dissociation", rtol=1e-2)
validate.close(
    E_R[-1],
    eps_1well,
    "E(R = 12) has reached the dissociation limit",
    rtol=0.0,
    atol=0.0011,  # one percent of the 0.1106 Ha well depth, matching the prose claim
)

## Exercise 5 — Why the separation works: two small numbers

The Born–Oppenheimer approximation is not an act of faith; it is controlled by
measurable ratios, and this model is small enough to measure both. First the *scale
separation*: the nuclei vibrate in the well of {numref}`fig-many-electron-bo` with
frequency $\omega = \sqrt{k/\mu}$, where $k = E''(R_{\mathrm{eq}})$ and
$\mu = M_p/2 = 918$ electron masses is the reduced mass of the proton pair, while the
electron's characteristic energy is the gap
$\Delta\varepsilon = \varepsilon_1 - \varepsilon_0$. Their ratio should be of order
$(m_e/\mu)^{1/2} = (2m_e/M_p)^{1/2} \approx 0.033$: nuclear timescales thirty times
slower. Second, the
*direct coupling*: the terms Born–Oppenheimer discards involve the response of the
electronic state to nuclear displacement, $\partial\phi_0/\partial R$, and their
energy scale

```{math}
:label: eq-me-nac
\delta E_{\mathrm{nac}}
= \frac{1}{2\mu}
\left\langle \frac{\partial\phi_0}{\partial R}
\middle| \frac{\partial\phi_0}{\partial R} \right\rangle
```

(the diagonal second-order coupling; Martin {cite}`martin2004`, Ch. 3, assembles the
full matrix) should be *tiny* against the gap.

**Part a)** The curvature $k = 2c_2$ is already in hand from the Exercise 4 parabola
fit. Compute $\omega = \sqrt{k/\mu}$ with $\mu = M_p/2$ from the imported
proton–electron mass ratio, the gap $\Delta\varepsilon$ at the grid point nearest
$R_{\mathrm{eq}}$, and the ratio $\omega/\Delta\varepsilon$.

**Part b)** Compute $\partial\phi_0/\partial R$ at $R_{\mathrm{eq}}$ by the central
difference of the normalized ground states at $R_{\mathrm{eq}} \pm 10^{-3}$ (align
their signs via the `numpy.dot` overlap before differencing: eigenvectors carry an
arbitrary sign), assemble Eq. {eq}`eq-me-nac` with a `numpy.dot` and the grid
spacing, and report $\delta E_{\mathrm{nac}}/\Delta\varepsilon$.

In [ ]:
# (solution hidden on the public site)


### Validation 5 — the separation is quantitatively excellent

The scale ratio must land near the mass-ratio prediction (within a factor accounting
for the well's stiffness), and the direct coupling must be smaller than the
electronic gap by better than four orders of magnitude: the two measured reasons the
rest of this volume may clamp its nuclei with a clear conscience.

In [ ]:
validate.close(ratio_scales, 0.031, "vibrational-to-electronic scale ratio", rtol=0.1)
validate.check(
    E_nac / gap_elec < 1e-4,
    "the discarded non-adiabatic coupling is negligible against the gap",
    f"ratio = {E_nac / gap_elec:.2e} (measured 6.8e-05 on this grid)",
)

## Exercise 6 — Helium, and the gap with a name

One last preparation, and the volume's antagonist steps on stage. Helium is the
smallest system where electrons must share space, and the oldest quantitative attack
on it is the screened product trial state
$\Phi_Z(\mathbf r_1, \mathbf r_2) = \tfrac{Z^3}{\pi} e^{-Z r_1} e^{-Z r_2}$: both
electrons in a hydrogenic $1s$ orbital of *adjustable* charge $Z$. The three
expectation values are classic closed forms (kinetic $Z^2$, nuclear attraction
$-4Z$ for the true charge $2$, and the electron–electron repulsion $\tfrac58 Z$;
Szabo & Ostlund {cite}`szabo1996`, §1.3, carry out the integrals), which sum to
Eq. {eq}`eq-me-hylleraas`, $E(Z) = Z^2 - \tfrac{27}{8}Z$. The variational principle
of [§6.22](../06-quantum-mechanics/variational-method.ipynb) guarantees every value
of this curve lies above the true ground energy; minimizing over $Z$ gives the best
the product form can do.

**Part a)** Minimize $E(Z)$ of Eq. {eq}`eq-me-hylleraas` with
`scipy.optimize.minimize_scalar` (`method="bounded"` on $Z \in [1, 2]$) and confirm
the closed-form optimum $Z_{\mathrm{eff}} = 27/16$ by setting $E'(Z) = 0$ by hand.
Report $E(Z_{\mathrm{eff}}) = -(27/16)^2 = -2.84766$ Ha.

**Part b)** Compare with the ladder of helium energies: the naive unscreened guess
$E(Z{=}2) = -2.75$, the screened optimum $-2.84766$, the Hartree–Fock limit
$-2.8617$ (the best *any* product state will achieve, computed in
[§8.3](hartree-fock-atoms.ipynb)), and the exact non-relativistic $-2.90372$
{cite}`parryang1989`. Plot $E(Z)$ with all four marked. The distance from the
Hartree–Fock limit to the exact energy, $-0.0420$ Ha, is the **correlation energy**
of helium — quote it in eV too, using the Hartree-to-eV factor certified in
Exercise 1: the part of the physics no one-electron picture can hold, and the
quantity Volume VIII's remaining notebooks are built to capture.

In [ ]:
# (solution hidden on the public site)


### Validation 6 — the optimum, the bound, and the gap

The numerical minimum must sit at $27/16$ with energy $-(27/16)^2$; every point of
the variational curve must lie above the exact energy (the variational bound); and
the correlation energy must come out $-0.042$ Ha.

In [ ]:
validate.close(res_he.x, 27.0 / 16.0, "the screened optimum Z_eff", rtol=1e-5)
validate.close(
    E_var, -((27.0 / 16.0) ** 2), "E at the optimum equals -(27/16)^2", rtol=1e-12
)
validate.check(
    bool(np.all(E_of_Z(Z_plot) > E_EXACT_HE)),
    "the variational bound holds along the whole curve",
    f"min over the curve = {E_of_Z(Z_plot).min():.5f} > exact {E_EXACT_HE}",
)
validate.close(E_corr, -0.042, "the correlation energy of helium", rtol=2e-2)

:::{admonition} With your assistant
:class: tip
The Born–Oppenheimer loop of Exercise 4 (a diagonalization per clamped geometry) is
exactly the kind of structured, repetitive code an assistant generates well. Have it
write a version that also returns the *third* electronic surface, then run the check
that is yours alone: at large $R$ the first and second surfaces must become
degenerate (the electron in the left or right well, split only by tunnelling), so
$\varepsilon_1 - \varepsilon_0$ at $R = 12$ must be smaller than $10^{-3}$ Ha while
$\varepsilon_2$ stays well separated. The check is yours.
:::

## Notebook summary

The problem is posed, and its difficulty is now a set of measured numbers rather than
a mood. Atomic units, built from the CODATA constants by direct arithmetic, reproduce
$a_0$ and $E_h = 27.2114$ eV to ten digits and strip every equation of the volume to
its physics. The radial tridiagonal machinery reproduced hydrogen's $-1/2$ and
$-1/8$ Ha and $\langle r\rangle = 3/2$ on the stated grid, so its later outputs on
helium and beryllium can be trusted. The exact wavefunction's storage cost climbs
$6.9$ decades per electron and passes the RAM of the entire planet between $N = 3$
and $N = 4$: the exact route is closed, permanently. The Born–Oppenheimer separation,
executed on a diagonalizable model molecule, produced a bound curve
($R_{\mathrm{eq}} = 2.62$ Bohr, binding $-0.111$ Ha) whose dissociation limit
vindicated the clamped-nuclei bookkeeping, and the separation's two error scales were
measured at $\omega/\Delta\varepsilon = 0.031$ and
$\delta E_{\mathrm{nac}}/\Delta\varepsilon = 7\times10^{-5}$. Helium's screened
product state minimized in closed form to $Z_{\mathrm{eff}} = 27/16$ and
$E = -2.8477$ Ha, above the exact $-2.9037$ by a gap no product state can close; the
distance from the Hartree–Fock limit, $-0.042$ Ha, received its name: correlation.

## Outlook

- The wall says the exact problem is unreachable in general, but not always: for two
  electrons in one dimension the full wavefunction fits comfortably in memory.
  [§8.2](exact-laboratory.ipynb) builds exactly that system and solves it *exactly*,
  giving the volume a referee every approximation must face.
- The screened-charge idea (each electron in an effective one-electron potential
  shaped by the others) is not a trick but a program. Made self-consistent, it
  becomes Hartree–Fock ([§8.3](hartree-fock-atoms.ipynb)); made exact in a way no
  one expected, it becomes Kohn–Sham density-functional theory
  ([§8.7](kohn-sham.ipynb)).
- The Born–Oppenheimer surfaces computed here for one electron become, for many, the
  central objects of chemistry: reaction barriers, vibrational spectra, molecular
  dynamics. The companion MMM course drives production codes across exactly such
  surfaces; this volume builds the machinery that computes them.

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()